In [ ]:
# If needed:
# !pip install codecarbon psutil

import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "6"  # physical GPU id; change if needed

import gc
import time
import random
import warnings
import logging
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
logging.getLogger("codecarbon").setLevel(logging.ERROR)
logging.getLogger("pynvml").setLevel(logging.ERROR)
logging.getLogger("urllib3").setLevel(logging.ERROR)

import torch
from codecarbon import EmissionsTracker
from bstabdiff_gobs import FeatureSpec, fit_block_subunit_generator


# ============================================================
# Config
# ============================================================
GLOBAL_SEED = 42
DEVICE = "cuda:0"  # physical GPU 6 becomes cuda:0 after masking
OUTPUT_DIR = "bstabdiff_energy_codecarbon"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DATASETS = {
    "COL": {"csv": "coloncancer_encoded.csv", "target": "label"},
    "LNG": {"csv": "lung_combined_encoded.csv", "target": "Label"},
    "GLI": {"csv": "GLI-85_encoded.csv", "target": "label"},
    "SMK": {"csv": "SMK_CAN_187_combined_encoded.csv", "target": "Label"},
    "AML": {"csv": "ALLAML_combined_encoded.csv", "target": "Label"},
    "PRS": {"csv": "Prostate-GE_combined_encoded.csv", "target": "Label"},
    "ARC": {"csv": "arcene_combined_encoded.csv", "target": "Label"},
    "TOX": {"csv": "TOX171_combined_encoded.csv", "target": "Label"},
}


BEST_PARAMS = {
    "COL": dict(M=8, prior_type="flow", prior_epochs=300, prior_batch=64,
                prior_lr=3.2931446587915556e-03, ema_decay=0.9925524818123974),
    "LNG": dict(M=8, prior_type="diffusion", prior_epochs=1200, prior_batch=128,
                prior_lr=5.8095129501182766e-05, ema_decay=0.9997542452283767),
    "GLI": dict(M=8, prior_type="diffusion", prior_epochs=800, prior_batch=32,
                prior_lr=3.138571708275628e-04, ema_decay=0.9910930012968403),
    "SMK": dict(M=8, prior_type="diffusion", prior_epochs=1200, prior_batch=32,
                prior_lr=1.459493893378754e-04, ema_decay=0.9995275220921749),
    "AML": dict(M=8, prior_type="flow", prior_epochs=1600, prior_batch=32,
                prior_lr=4.9452480563480705e-03, ema_decay=0.9928183740289842),
    "PRS": dict(M=16, prior_type="diffusion", prior_epochs=1200, prior_batch=64,
                prior_lr=2.5615330390421176e-04, ema_decay=0.9982816164147802),
    "ARC": dict(M=8, prior_type="diffusion", prior_epochs=1200, prior_batch=16,
                prior_lr=1.6478907059217304e-04, ema_decay=0.998970675192287),
    "TOX": dict(M=16, prior_type="diffusion", prior_epochs=300, prior_batch=64,
                prior_lr=5.81932245081814e-04, ema_decay=0.9990556532314717),
}


# ============================================================
# Helpers
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_dataset(csv_path, target_col):
    df = pd.read_csv(csv_path)

    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in {csv_path}.")

    y_raw = df[target_col].values
    X_df = df.drop(columns=[target_col]).select_dtypes(include=[np.number])

    X = X_df.to_numpy(dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    if y_raw.dtype.kind in {"U", "S", "O"}:
        y, _ = pd.factorize(y_raw)
    else:
        uniq = np.unique(y_raw)
        if not np.array_equal(uniq, np.arange(len(uniq))):
            y, _ = pd.factorize(y_raw)
        else:
            y = y_raw.astype(int)

    return X, y.astype(int), list(X_df.columns)


def train_bstabdiff(X, y, params, name):
    feature_specs = [
        FeatureSpec(name=f"f{j}", kind="continuous")
        for j in range(X.shape[1])
    ]

    gen = fit_block_subunit_generator(
        X=X,
        feature_specs=feature_specs,
        y=y,
        M=params["M"],
        blocks=None,
        permute_features=False,
        prior_type=params["prior_type"],
        device=DEVICE,
        seed=GLOBAL_SEED,
        prior_epochs=params["prior_epochs"],
        prior_batch=params["prior_batch"],
        prior_lr=params["prior_lr"],
        verbose_every=0,
        save_dir=None,
        save_name=f"bstabdiff_energy_{name}",
        save_best=True,
        use_ema=True,
        ema_decay=params["ema_decay"],
        return_train_info=False,

        # fixed no-GO-BS / no-permutation setup
        use_gobs=False,
        gobs_num_clusters=7,
        gobs_metric="correlation",
        gobs_bins=32,
        gobs_top_k=None,
        gobs_refine_order=False,
        gobs_direction_select=False,
        gobs_refine_passes=0,
        gobs_boundary_refine_passes=1,
        gobs_boundary_window=8,
        gobs_lambda_cross=1.0,
        gobs_gamma_balance=1e-6,
        gobs_use_cpu_kmeans=False,
    )
    return gen


def read_codecarbon_last_row(csv_path):
    df = pd.read_csv(csv_path)
    row = df.iloc[-1].to_dict()

    def get(*keys, default=np.nan):
        for k in keys:
            if k in row:
                return row[k]
        return default

    return {
        "energy_kwh": get("energy_consumed"),
        "emissions_kg_co2e": get("emissions"),
        "cpu_energy_kwh": get("cpu_energy"),
        "gpu_energy_kwh": get("gpu_energy"),
        "ram_energy_kwh": get("ram_energy"),
        "cpu_power_w": get("cpu_power"),
        "gpu_power_w": get("gpu_power"),
        "ram_power_w": get("ram_power"),
    }


# ============================================================
# Run CodeCarbon tracking
# ============================================================
set_seed(GLOBAL_SEED)

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.cuda.device_count():", torch.cuda.device_count())

if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))

all_rows = []

# To test only two datasets first, replace DATASETS.items() with:
# {k: DATASETS[k] for k in ["COL", "LNG"]}.items()

for name, cfg in DATASETS.items():
    print(f"\n{'=' * 80}")
    print(f"Dataset: {name}")
    print(f"{'=' * 80}")

    X, y, feature_names = load_dataset(cfg["csv"], cfg["target"])
    print(f"Loaded {name}: X={X.shape}, classes={np.unique(y)}")

    params = BEST_PARAMS[name]
    cleanup()

    output_file = f"codecarbon_{name}.csv"
    output_path = os.path.join(OUTPUT_DIR, output_file)

    tracker = EmissionsTracker(
        project_name=f"BSTabDiff_{name}",
        output_dir=OUTPUT_DIR,
        output_file=output_file,
        measure_power_secs=1,
        save_to_file=True,
        log_level="error",
    )

    t0 = time.time()
    tracker.start()

    try:
        _ = train_bstabdiff(X, y, params, name)
    finally:
        emissions = tracker.stop()
        elapsed = time.time() - t0
        cleanup()

    cc = read_codecarbon_last_row(output_path)

    row = {
        "Dataset": name,
        "n": X.shape[0],
        "m": X.shape[1],
        "classes": len(np.unique(y)),
        "energy_kwh": cc["energy_kwh"],
        "emissions_kg_co2e": cc["emissions_kg_co2e"],
        "cpu_energy_kwh": cc["cpu_energy_kwh"],
        "gpu_energy_kwh": cc["gpu_energy_kwh"],
        "ram_energy_kwh": cc["ram_energy_kwh"],
        "cpu_power_w": cc["cpu_power_w"],
        "gpu_power_w": cc["gpu_power_w"],
        "ram_power_w": cc["ram_power_w"],
        "elapsed_sec": elapsed,
        "codecarbon_returned_emissions": emissions,
    }

    all_rows.append(row)

    print(
        f"{name}: energy={row['energy_kwh']:.6f} kWh, "
        f"CO2e={row['emissions_kg_co2e']:.6f} kg, "
        f"time={elapsed:.2f}s"
    )

out_df = pd.DataFrame(all_rows)
out_csv = os.path.join(OUTPUT_DIR, "bstabdiff_energy_co2_summary.csv")
out_df.to_csv(out_csv, index=False)

print("\nSaved summary:")
print(out_csv)

try:
    display(out_df)
except Exception:
    print(out_df.to_markdown(index=False))

CUDA_VISIBLE_DEVICES: 6
torch.cuda.is_available(): True
torch.cuda.device_count(): 1
Using GPU: NVIDIA RTX A6000

Dataset: COL
Loaded COL: X=(62, 2000), classes=[0 1]


[codecarbon WARNING @ 23:43:10] Multiple instances of codecarbon are allowed to run at the same time.


COL: energy=0.000479 kWh, CO2e=0.000429 kg, time=12.00s

Dataset: LNG
Loaded LNG: X=(203, 3312), classes=[0 1 2 3 4]
LNG: energy=0.000317 kWh, CO2e=0.000284 kg, time=7.23s

Dataset: GLI
Loaded GLI: X=(85, 22283), classes=[0 1]
GLI: energy=0.000369 kWh, CO2e=0.000331 kg, time=11.05s

Dataset: SMK
Loaded SMK: X=(187, 19993), classes=[0 1]
SMK: energy=0.000468 kWh, CO2e=0.000420 kg, time=11.50s

Dataset: AML
Loaded AML: X=(72, 7129), classes=[0 1]
AML: energy=0.001309 kWh, CO2e=0.001173 kg, time=33.82s

Dataset: PRS
Loaded PRS: X=(102, 5966), classes=[0 1]
PRS: energy=0.000252 kWh, CO2e=0.000226 kg, time=7.48s

Dataset: ARC
Loaded ARC: X=(200, 10000), classes=[0 1]
ARC: energy=0.000430 kWh, CO2e=0.000385 kg, time=8.55s

Dataset: TOX
Loaded TOX: X=(171, 5748), classes=[0 1 2 3]
TOX: energy=0.000192 kWh, CO2e=0.000172 kg, time=5.11s

Saved summary:
bstabdiff_energy_codecarbon/bstabdiff_energy_co2_summary.csv


,Dataset,n,m,classes,energy_kwh,emissions_kg_co2e,cpu_energy_kwh,gpu_energy_kwh,ram_energy_kwh,cpu_power_w,gpu_power_w,ram_power_w,elapsed_sec,codecarbon_returned_emissions
0,COL,62,2000,2,0.000479,0.000429,0.000094,0.000212,0.000173,37.763306,152.973336,70.0,12.002296,0.000429
1,LNG,203,3312,5,0.000317,0.000284,0.000061,0.000132,0.000124,31.145155,86.339679,70.0,7.234746,0.000284
2,GLI,85,22283,2,0.000369,0.000331,0.000052,0.000203,0.000114,33.949900,93.248284,70.0,11.052350,0.000331
3,SMK,187,19993,2,0.000468,0.000420,0.000088,0.000216,0.000164,37.576924,99.719041,70.0,11.496838,0.000420
4,AML,72,7129,2,0.001309,0.001173,0.000216,0.000694,0.000399,37.941593,103.954609,70.0,33.821958,0.001173
5,PRS,102,5966,2,0.000252,0.000226,0.000043,0.000130,0.000079,37.948064,75.128284,70.0,7.480723,0.000226
6,ARC,200,10000,2,0.000430,0.000385,0.000097,0.000154,0.000179,37.547319,136.324242,70.0,8.552372,0.000385
7,TOX,171,5748,4,0.000192,0.000172,0.000035,0.000084,0.000073,28.265106,94.181929,70.0,5.109746,0.000172


**The above code is for tracking carbon/energy measures for BSTabDiff without GO-BS**

# w/ GO-BS

In [ ]:
# If needed:
# !pip install codecarbon psutil

import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

import gc
import time
import random
import warnings
import logging
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
logging.getLogger("codecarbon").setLevel(logging.ERROR)
logging.getLogger("pynvml").setLevel(logging.ERROR)
logging.getLogger("urllib3").setLevel(logging.ERROR)

import torch
from codecarbon import EmissionsTracker
from bstabdiff_gobs import FeatureSpec, fit_block_subunit_generator


GLOBAL_SEED = 42
DEVICE = "cuda:0"
OUTPUT_DIR = "bstabdiff_energy_codecarbon_gobs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DATASETS = {
    "COL": {"csv": "coloncancer_encoded.csv", "target": "label"},
    "LNG": {"csv": "lung_combined_encoded.csv", "target": "Label"},
    "GLI": {"csv": "GLI-85_encoded.csv", "target": "label"},
    "SMK": {"csv": "SMK_CAN_187_combined_encoded.csv", "target": "Label"},
    "AML": {"csv": "ALLAML_combined_encoded.csv", "target": "Label"},
    "PRS": {"csv": "Prostate-GE_combined_encoded.csv", "target": "Label"},
    "ARC": {"csv": "arcene_combined_encoded.csv", "target": "Label"},
    "TOX": {"csv": "TOX171_combined_encoded.csv", "target": "Label"},
}

# Main BSTabDiff tuned params, same as no-GO-BS run
BEST_PARAMS = {
    "COL": dict(M=8, prior_type="flow", prior_epochs=300, prior_batch=64,
                prior_lr=3.2931446587915556e-03, ema_decay=0.9925524818123974),
    "LNG": dict(M=8, prior_type="diffusion", prior_epochs=1200, prior_batch=128,
                prior_lr=5.8095129501182766e-05, ema_decay=0.9997542452283767),
    "GLI": dict(M=8, prior_type="diffusion", prior_epochs=800, prior_batch=32,
                prior_lr=3.138571708275628e-04, ema_decay=0.9910930012968403),
    "SMK": dict(M=8, prior_type="diffusion", prior_epochs=1200, prior_batch=32,
                prior_lr=1.459493893378754e-04, ema_decay=0.9995275220921749),
    "AML": dict(M=8, prior_type="flow", prior_epochs=1600, prior_batch=32,
                prior_lr=4.9452480563480705e-03, ema_decay=0.9928183740289842),
    "PRS": dict(M=16, prior_type="diffusion", prior_epochs=1200, prior_batch=64,
                prior_lr=2.5615330390421176e-04, ema_decay=0.9982816164147802),
    "ARC": dict(M=8, prior_type="diffusion", prior_epochs=1200, prior_batch=16,
                prior_lr=1.6478907059217304e-04, ema_decay=0.998970675192287),
    "TOX": dict(M=16, prior_type="diffusion", prior_epochs=300, prior_batch=64,
                prior_lr=5.81932245081814e-04, ema_decay=0.9990556532314717),
}

# GO-BS-only tuned params
GOBS_PARAMS = {
    "COL": dict(gobs_num_clusters=3, gobs_metric="correlation", gobs_bins=24,
                gobs_top_k=16, gobs_refine_order=False, gobs_direction_select=False,
                gobs_refine_passes=3, gobs_boundary_refine_passes=5,
                gobs_boundary_window=16, gobs_lambda_cross=2.3988509383361634,
                gobs_gamma_balance=5.193972052958215e-06),

    "LNG": dict(gobs_num_clusters=3, gobs_metric="cosine", gobs_bins=32,
                gobs_top_k=16, gobs_refine_order=True, gobs_direction_select=False,
                gobs_refine_passes=1, gobs_boundary_refine_passes=5,
                gobs_boundary_window=16, gobs_lambda_cross=1.1691157379725399,
                gobs_gamma_balance=2.649232087240688e-08),

    "GLI": dict(gobs_num_clusters=3, gobs_metric="manhattan", gobs_bins=32,
                gobs_top_k=32, gobs_refine_order=True, gobs_direction_select=False,
                gobs_refine_passes=1, gobs_boundary_refine_passes=0,
                gobs_boundary_window=24, gobs_lambda_cross=2.4372992085995695,
                gobs_gamma_balance=9.273654124034148e-06),

    "SMK": dict(gobs_num_clusters=7, gobs_metric="correlation", gobs_bins=64,
                gobs_top_k=64, gobs_refine_order=True, gobs_direction_select=True,
                gobs_refine_passes=2, gobs_boundary_refine_passes=5,
                gobs_boundary_window=12, gobs_lambda_cross=0.3031628494242844,
                gobs_gamma_balance=2.684956739461984e-07),

    "AML": dict(gobs_num_clusters=3, gobs_metric="correlation", gobs_bins=16,
                gobs_top_k=16, gobs_refine_order=False, gobs_direction_select=True,
                gobs_refine_passes=2, gobs_boundary_refine_passes=8,
                gobs_boundary_window=8, gobs_lambda_cross=0.16065998434856013,
                gobs_gamma_balance=1.8574882089019752e-05),

    "PRS": dict(gobs_num_clusters=3, gobs_metric="kl_divergence", gobs_bins=24,
                gobs_top_k=16, gobs_refine_order=True, gobs_direction_select=True,
                gobs_refine_passes=0, gobs_boundary_refine_passes=5,
                gobs_boundary_window=12, gobs_lambda_cross=0.18922945545256775,
                gobs_gamma_balance=3.7831418714171236e-05),

    "ARC": dict(gobs_num_clusters=7, gobs_metric="cosine", gobs_bins=48,
                gobs_top_k=32, gobs_refine_order=True, gobs_direction_select=False,
                gobs_refine_passes=2, gobs_boundary_refine_passes=5,
                gobs_boundary_window=12, gobs_lambda_cross=2.750375511297408,
                gobs_gamma_balance=0.0007575070435534387),

    "TOX": dict(gobs_num_clusters=4, gobs_metric="correlation", gobs_bins=48,
                gobs_top_k=64, gobs_refine_order=False, gobs_direction_select=False,
                gobs_refine_passes=1, gobs_boundary_refine_passes=3,
                gobs_boundary_window=16, gobs_lambda_cross=2.3373024863461076,
                gobs_gamma_balance=3.249922004941342e-08),
}


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_dataset(csv_path, target_col):
    df = pd.read_csv(csv_path)

    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in {csv_path}.")

    y_raw = df[target_col].values
    X_df = df.drop(columns=[target_col]).select_dtypes(include=[np.number])

    X = X_df.to_numpy(dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    if y_raw.dtype.kind in {"U", "S", "O"}:
        y, _ = pd.factorize(y_raw)
    else:
        uniq = np.unique(y_raw)
        if not np.array_equal(uniq, np.arange(len(uniq))):
            y, _ = pd.factorize(y_raw)
        else:
            y = y_raw.astype(int)

    return X, y.astype(int), list(X_df.columns)


def train_bstabdiff_gobs(X, y, params, gobs_params, name):
    feature_specs = [
        FeatureSpec(name=f"f{j}", kind="continuous")
        for j in range(X.shape[1])
    ]

    gen = fit_block_subunit_generator(
        X=X,
        feature_specs=feature_specs,
        y=y,
        M=params["M"],
        blocks=None,
        permute_features=False,
        prior_type=params["prior_type"],
        device=DEVICE,
        seed=GLOBAL_SEED,
        prior_epochs=params["prior_epochs"],
        prior_batch=params["prior_batch"],
        prior_lr=params["prior_lr"],
        verbose_every=0,
        save_dir=None,
        save_name=f"bstabdiff_energy_gobs_{name}",
        save_best=True,
        use_ema=True,
        ema_decay=params["ema_decay"],
        return_train_info=False,

        # GO-BS enabled
        use_gobs=True,
        gobs_num_clusters=gobs_params["gobs_num_clusters"],
        gobs_metric=gobs_params["gobs_metric"],
        gobs_bins=gobs_params["gobs_bins"],
        gobs_top_k=gobs_params["gobs_top_k"],
        gobs_refine_order=gobs_params["gobs_refine_order"],
        gobs_direction_select=gobs_params["gobs_direction_select"],
        gobs_refine_passes=gobs_params["gobs_refine_passes"],
        gobs_boundary_refine_passes=gobs_params["gobs_boundary_refine_passes"],
        gobs_boundary_window=gobs_params["gobs_boundary_window"],
        gobs_lambda_cross=gobs_params["gobs_lambda_cross"],
        gobs_gamma_balance=gobs_params["gobs_gamma_balance"],
        gobs_use_cpu_kmeans=False,
    )
    return gen


def read_codecarbon_last_row(csv_path):
    df = pd.read_csv(csv_path)
    row = df.iloc[-1].to_dict()

    def get(*keys, default=np.nan):
        for k in keys:
            if k in row:
                return row[k]
        return default

    return {
        "energy_kwh": get("energy_consumed"),
        "emissions_kg_co2e": get("emissions"),
        "cpu_energy_kwh": get("cpu_energy"),
        "gpu_energy_kwh": get("gpu_energy"),
        "ram_energy_kwh": get("ram_energy"),
        "cpu_power_w": get("cpu_power"),
        "gpu_power_w": get("gpu_power"),
        "ram_power_w": get("ram_power"),
    }


set_seed(GLOBAL_SEED)

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.cuda.device_count():", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))

all_rows = []

for name, cfg in DATASETS.items():
    print(f"\n{'=' * 80}")
    print(f"Dataset: {name} | BSTabDiff + GO-BS")
    print(f"{'=' * 80}")

    X, y, feature_names = load_dataset(cfg["csv"], cfg["target"])
    print(f"Loaded {name}: X={X.shape}, classes={np.unique(y)}")

    params = BEST_PARAMS[name]
    gobs_params = GOBS_PARAMS[name]
    cleanup()

    output_file = f"codecarbon_gobs_{name}.csv"
    output_path = os.path.join(OUTPUT_DIR, output_file)

    tracker = EmissionsTracker(
        project_name=f"BSTabDiff_GO_BS_{name}",
        output_dir=OUTPUT_DIR,
        output_file=output_file,
        measure_power_secs=1,
        save_to_file=True,
        log_level="error",
    )

    t0 = time.time()
    tracker.start()

    try:
        _ = train_bstabdiff_gobs(X, y, params, gobs_params, name)
    finally:
        emissions = tracker.stop()
        elapsed = time.time() - t0
        cleanup()

    cc = read_codecarbon_last_row(output_path)

    row = {
        "Dataset": name,
        "n": X.shape[0],
        "m": X.shape[1],
        "classes": len(np.unique(y)),
        "energy_kwh": cc["energy_kwh"],
        "emissions_kg_co2e": cc["emissions_kg_co2e"],
        "cpu_energy_kwh": cc["cpu_energy_kwh"],
        "gpu_energy_kwh": cc["gpu_energy_kwh"],
        "ram_energy_kwh": cc["ram_energy_kwh"],
        "cpu_power_w": cc["cpu_power_w"],
        "gpu_power_w": cc["gpu_power_w"],
        "ram_power_w": cc["ram_power_w"],
        "elapsed_sec": elapsed,
        "codecarbon_returned_emissions": emissions,
    }

    all_rows.append(row)

    print(
        f"{name}: energy={row['energy_kwh']:.6f} kWh, "
        f"CO2e={row['emissions_kg_co2e']:.6f} kg, "
        f"time={elapsed:.2f}s"
    )

out_df = pd.DataFrame(all_rows)
out_csv = os.path.join(OUTPUT_DIR, "bstabdiff_gobs_energy_co2_summary.csv")
out_df.to_csv(out_csv, index=False)

print("\nSaved summary:")
print(out_csv)

try:
    display(out_df)
except Exception:
    print(out_df.to_markdown(index=False))

CUDA_VISIBLE_DEVICES: 6
torch.cuda.is_available(): True
torch.cuda.device_count(): 1
Using GPU: NVIDIA RTX A6000

Dataset: COL | BSTabDiff + GO-BS
Loaded COL: X=(62, 2000), classes=[0 1]
COL: energy=0.000577 kWh, CO2e=0.000517 kg, time=15.37s

Dataset: LNG | BSTabDiff + GO-BS
Loaded LNG: X=(203, 3312), classes=[0 1 2 3 4]
LNG: energy=0.000690 kWh, CO2e=0.000618 kg, time=19.48s

Dataset: GLI | BSTabDiff + GO-BS
Loaded GLI: X=(85, 22283), classes=[0 1]
GLI: energy=0.003113 kWh, CO2e=0.002790 kg, time=56.48s

Dataset: SMK | BSTabDiff + GO-BS
Loaded SMK: X=(187, 19993), classes=[0 1]
SMK: energy=0.007191 kWh, CO2e=0.006444 kg, time=145.81s

Dataset: AML | BSTabDiff + GO-BS
Loaded AML: X=(72, 7129), classes=[0 1]
AML: energy=0.001768 kWh, CO2e=0.001584 kg, time=44.79s

Dataset: PRS | BSTabDiff + GO-BS
Loaded PRS: X=(102, 5966), classes=[0 1]
PRS: energy=0.002178 kWh, CO2e=0.001952 kg, time=41.91s

Dataset: ARC | BSTabDiff + GO-BS
Loaded ARC: X=(200, 10000), classes=[0 1]
ARC: energy=0.00271

,Dataset,n,m,classes,energy_kwh,emissions_kg_co2e,cpu_energy_kwh,gpu_energy_kwh,ram_energy_kwh,cpu_power_w,gpu_power_w,ram_power_w,elapsed_sec,codecarbon_returned_emissions
0,COL,62,2000,2,0.000577,0.000517,0.000088,0.000326,0.000163,37.687078,103.192911,70.0,15.367781,0.000517
1,LNG,203,3312,5,0.000690,0.000618,0.000083,0.000450,0.000157,35.719919,116.078091,70.0,19.483034,0.000618
2,GLI,85,22283,2,0.003113,0.002790,0.000256,0.002380,0.000477,37.551630,202.293833,70.0,56.476748,0.002790
3,SMK,187,19993,2,0.007191,0.006444,0.000690,0.005215,0.001286,37.549220,181.287722,70.0,145.808341,0.006444
4,AML,72,7129,2,0.001768,0.001584,0.000187,0.001232,0.000349,37.595160,131.912681,70.0,44.786797,0.001584
5,PRS,102,5966,2,0.002178,0.001952,0.000221,0.001547,0.000410,37.693406,231.686670,70.0,41.912846,0.001952
6,ARC,200,10000,2,0.002712,0.002430,0.000315,0.001810,0.000587,37.547552,148.199404,70.0,61.276848,0.002430
7,TOX,171,5748,4,0.001726,0.001547,0.000189,0.001187,0.000351,37.514324,218.930163,70.0,32.932484,0.001547


# w/ GO-BS-FC

In [ ]:
# If needed:
# !pip install codecarbon psutil kmeans-gpu

import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

import gc
import time
import random
import warnings
import logging
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
logging.getLogger("codecarbon").setLevel(logging.ERROR)
logging.getLogger("pynvml").setLevel(logging.ERROR)
logging.getLogger("urllib3").setLevel(logging.ERROR)

import torch
from codecarbon import EmissionsTracker

try:
    from kmeans_gpu import KMeans as KMeansGPU
except Exception:
    KMeansGPU = None

from bstabdiff_gobs import FeatureSpec, fit_block_subunit_generator


# ============================================================
# Config
# ============================================================
GLOBAL_SEED = 42
DEVICE = "cuda:0"
OUTPUT_DIR = "bstabdiff_energy_codecarbon_gobs_fc"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DATASETS = {
    "COL": {"csv": "coloncancer_encoded.csv", "target": "label"},
    "LNG": {"csv": "lung_combined_encoded.csv", "target": "Label"},
    "GLI": {"csv": "GLI-85_encoded.csv", "target": "label"},
    "SMK": {"csv": "SMK_CAN_187_combined_encoded.csv", "target": "Label"},
    "AML": {"csv": "ALLAML_combined_encoded.csv", "target": "Label"},
    "PRS": {"csv": "Prostate-GE_combined_encoded.csv", "target": "Label"},
    "ARC": {"csv": "arcene_combined_encoded.csv", "target": "Label"},
    "TOX": {"csv": "TOX171_combined_encoded.csv", "target": "Label"},
}

BEST_PARAMS = {
    "COL": dict(M=8, prior_type="flow", prior_epochs=300, prior_batch=64,
                prior_lr=3.2931446587915556e-03, ema_decay=0.9925524818123974),
    "LNG": dict(M=8, prior_type="diffusion", prior_epochs=1200, prior_batch=128,
                prior_lr=5.8095129501182766e-05, ema_decay=0.9997542452283767),
    "GLI": dict(M=8, prior_type="diffusion", prior_epochs=800, prior_batch=32,
                prior_lr=3.138571708275628e-04, ema_decay=0.9910930012968403),
    "SMK": dict(M=8, prior_type="diffusion", prior_epochs=1200, prior_batch=32,
                prior_lr=1.459493893378754e-04, ema_decay=0.9995275220921749),
    "AML": dict(M=8, prior_type="flow", prior_epochs=1600, prior_batch=32,
                prior_lr=4.9452480563480705e-03, ema_decay=0.9928183740289842),
    "PRS": dict(M=16, prior_type="diffusion", prior_epochs=1200, prior_batch=64,
                prior_lr=2.5615330390421176e-04, ema_decay=0.9982816164147802),
    "ARC": dict(M=8, prior_type="diffusion", prior_epochs=1200, prior_batch=16,
                prior_lr=1.6478907059217304e-04, ema_decay=0.998970675192287),
    "TOX": dict(M=16, prior_type="diffusion", prior_epochs=300, prior_batch=64,
                prior_lr=5.81932245081814e-04, ema_decay=0.9990556532314717),
}

# Reuse GO-BS tuned graph parameters for FC ordering
GOBS_FC_PARAMS = {
    "COL": dict(gobs_num_clusters=3, gobs_metric="correlation", gobs_bins=24,
                gobs_top_k=16, gobs_refine_passes=3),
    "LNG": dict(gobs_num_clusters=3, gobs_metric="cosine", gobs_bins=32,
                gobs_top_k=16, gobs_refine_passes=1),
    "GLI": dict(gobs_num_clusters=3, gobs_metric="manhattan", gobs_bins=32,
                gobs_top_k=32, gobs_refine_passes=1),
    "SMK": dict(gobs_num_clusters=7, gobs_metric="correlation", gobs_bins=64,
                gobs_top_k=64, gobs_refine_passes=2),
    "AML": dict(gobs_num_clusters=3, gobs_metric="correlation", gobs_bins=16,
                gobs_top_k=16, gobs_refine_passes=2),
    "PRS": dict(gobs_num_clusters=3, gobs_metric="kl_divergence", gobs_bins=24,
                gobs_top_k=16, gobs_refine_passes=0),
    "ARC": dict(gobs_num_clusters=7, gobs_metric="cosine", gobs_bins=48,
                gobs_top_k=32, gobs_refine_passes=2),
    "TOX": dict(gobs_num_clusters=4, gobs_metric="correlation", gobs_bins=48,
                gobs_top_k=64, gobs_refine_passes=1),
}


# ============================================================
# Helpers
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_dataset(csv_path, target_col):
    df = pd.read_csv(csv_path)

    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in {csv_path}.")

    y_raw = df[target_col].values
    X_df = df.drop(columns=[target_col]).select_dtypes(include=[np.number])

    X = X_df.to_numpy(dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    if y_raw.dtype.kind in {"U", "S", "O"}:
        y, _ = pd.factorize(y_raw)
    else:
        uniq = np.unique(y_raw)
        if not np.array_equal(uniq, np.arange(len(uniq))):
            y, _ = pd.factorize(y_raw)
        else:
            y = y_raw.astype(int)

    return X, y.astype(int), list(X_df.columns)


def invert_permutation(order):
    order = np.asarray(order, dtype=int)
    inv = np.empty_like(order)
    inv[order] = np.arange(len(order))
    return inv


def make_contiguous_blocks_from_order(m, M):
    M = int(max(1, min(M, m)))
    base = m // M
    rem = m % M

    blocks = []
    start = 0
    for k in range(M):
        size = base + (1 if k < rem else 0)
        end = start + size
        if size > 0:
            blocks.append(np.arange(start, end, dtype=int))
        start = end
    return blocks


# ============================================================
# GO-BS-FC ordering helpers
# ============================================================
def safe_standardize_rows(X, eps=1e-8):
    X = np.asarray(X, dtype=np.float64)
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    sd = np.where(sd < eps, 1.0, sd)
    return (X - mu) / sd


def safe_standardize_cols(X, eps=1e-8):
    X = np.asarray(X, dtype=np.float64)
    mu = X.mean(axis=0, keepdims=True)
    sd = X.std(axis=0, keepdims=True)
    sd = np.where(sd < eps, 1.0, sd)
    return (X - mu) / sd


def require_gpu_kmeans():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for GO-BS-FC GPU KMeans.")
    if KMeansGPU is None:
        raise ImportError("kmeans-gpu is not installed. Install with: pip install kmeans-gpu")


def _run_kmeans_gpu_points(X_points, n_clusters, max_iter=100, tolerance=1e-4):
    require_gpu_kmeans()

    X_points = np.asarray(X_points, dtype=np.float32)
    n, d = X_points.shape
    n_clusters = int(max(1, min(n_clusters, n)))

    pts = torch.as_tensor(X_points, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    dummy_feat = torch.zeros((1, 1, n), dtype=torch.float32, device=DEVICE)

    kmeans = KMeansGPU(
        n_clusters=n_clusters,
        max_iter=max_iter,
        tolerance=tolerance,
        distance="euclidean",
        sub_sampling=None,
        max_neighbors=max(15, min(64, n)),
    ).to(DEVICE)

    with torch.no_grad():
        centroids, _ = kmeans(pts, dummy_feat)
        centroids = centroids[0]
        dists = torch.cdist(pts[0], centroids)
        labels = torch.argmin(dists, dim=1)

    return centroids.detach().cpu().numpy(), labels.detach().cpu().numpy().astype(int)


def build_feature_affinity(X_sub, metric="cosine", bins=64, eps=1e-8):
    X_sub = np.asarray(X_sub, dtype=np.float64)
    _, m = X_sub.shape

    if m == 1:
        return np.zeros((1, 1), dtype=np.float64)

    if metric == "correlation":
        Z = safe_standardize_cols(X_sub, eps=eps)
        W = np.corrcoef(Z, rowvar=False)
        W = np.nan_to_num(W, nan=0.0, posinf=0.0, neginf=0.0)
        W = np.abs(W)

    elif metric == "cosine":
        F = X_sub.T
        norms = np.linalg.norm(F, axis=1, keepdims=True)
        norms = np.where(norms < eps, 1.0, norms)
        F = F / norms
        W = F @ F.T
        W = np.nan_to_num(W, nan=0.0, posinf=0.0, neginf=0.0)
        W = np.abs(W)

    elif metric in ("euclidean", "manhattan"):
        F = X_sub.T
        W = np.zeros((m, m), dtype=np.float64)
        for i in range(m):
            diff = np.abs(F[i][None, :] - F)
            if metric == "euclidean":
                d = np.sqrt((diff ** 2).sum(axis=1))
            else:
                d = diff.sum(axis=1)
            W[i] = d
        scale = np.median(W[W > 0]) if np.any(W > 0) else 1.0
        scale = max(scale, eps)
        W = np.exp(-W / scale)

    elif metric == "kl_divergence":
        F = X_sub.T
        W = np.zeros((m, m), dtype=np.float64)
        for i in range(m):
            xi = F[i]
            for j in range(i, m):
                xj = F[j]
                lo = min(np.min(xi), np.min(xj))
                hi = max(np.max(xi), np.max(xj))
                if hi <= lo:
                    hi = lo + 1e-6
                hist_i, _ = np.histogram(xi, bins=bins, range=(lo, hi), density=True)
                hist_j, _ = np.histogram(xj, bins=bins, range=(lo, hi), density=True)
                p = hist_i + eps
                q = hist_j + eps
                p = p / p.sum()
                q = q / q.sum()
                skl = 0.5 * (np.sum(p * np.log(p / q)) + np.sum(q * np.log(q / p)))
                W[i, j] = np.exp(-skl)
                W[j, i] = W[i, j]
    else:
        raise ValueError(f"Unknown metric: {metric}")

    W = np.nan_to_num(W, nan=0.0, posinf=0.0, neginf=0.0)
    W = 0.5 * (W + W.T)
    np.fill_diagonal(W, 0.0)

    mx = np.max(W)
    if mx > 0:
        W = W / mx
    return W


def sparsify_topk(W, top_k=8):
    if top_k is None:
        return W

    W = np.asarray(W, dtype=np.float64)
    m = W.shape[0]
    K = int(min(max(1, top_k), max(1, m - 1)))

    Ws = np.zeros_like(W)
    for i in range(m):
        idx = np.argsort(W[i])[::-1]
        idx = idx[idx != i][:K]
        Ws[i, idx] = W[i, idx]

    Ws = np.maximum(Ws, Ws.T)
    np.fill_diagonal(Ws, 0.0)

    mx = np.max(Ws)
    if mx > 0:
        Ws = Ws / mx
    return Ws


def greedy_order_from_affinity(W, refine_passes=2):
    W = np.asarray(W, dtype=np.float64)
    m = W.shape[0]

    if m == 1:
        return np.array([0], dtype=int)

    deg = W.sum(axis=1)
    start = int(np.argmax(deg))

    order = [start]
    unvisited = set(range(m))
    unvisited.remove(start)

    while unvisited:
        last = order[-1]
        cand = max(unvisited, key=lambda j: (W[last, j], deg[j]))
        order.append(int(cand))
        unvisited.remove(cand)

    order = np.array(order, dtype=int)

    def score(ord_):
        if len(ord_) <= 1:
            return 0.0
        return sum(W[ord_[i], ord_[i + 1]] for i in range(len(ord_) - 1))

    for _ in range(refine_passes):
        improved = False
        for i in range(m - 1):
            cand = order.copy()
            cand[i], cand[i + 1] = cand[i + 1], cand[i]
            if score(cand) > score(order):
                order = cand
                improved = True
        if not improved:
            break

    return order


def build_feature_clustered_order_gpu(
    X,
    num_clusters=5,
    metric="cosine",
    bins=64,
    top_k=8,
    refine_passes=3,
):
    X = np.asarray(X, dtype=np.float64)
    n, m = X.shape

    F = X.T
    Fz = safe_standardize_rows(F)

    C = int(max(2, min(num_clusters, m)))
    _, feat_labels = _run_kmeans_gpu_points(Fz, n_clusters=C)

    clusters = []
    for c in range(C):
        idx = np.where(feat_labels == c)[0]
        if len(idx) > 0:
            clusters.append(np.asarray(idx, dtype=int))

    cluster_internal_orders = []
    cluster_reps = []

    for feat_idx in clusters:
        Xc = X[:, feat_idx]

        if len(feat_idx) == 1:
            local_order = np.array([0], dtype=int)
            rep = X[:, feat_idx[0]]
        else:
            Wc = build_feature_affinity(Xc, metric=metric, bins=bins)
            Wc = sparsify_topk(Wc, top_k=min(top_k, max(1, len(feat_idx) - 1)))
            local_order = greedy_order_from_affinity(Wc, refine_passes=refine_passes)
            rep = Xc[:, local_order].mean(axis=1)

        cluster_internal_orders.append(feat_idx[local_order])
        cluster_reps.append(rep)

    R = np.column_stack(cluster_reps)
    W_cluster = build_feature_affinity(R, metric=metric, bins=bins)
    W_cluster = sparsify_topk(W_cluster, top_k=min(top_k, max(1, W_cluster.shape[0] - 1)))
    cluster_order = greedy_order_from_affinity(W_cluster, refine_passes=refine_passes)

    final_order = np.concatenate([cluster_internal_orders[c] for c in cluster_order]).astype(int)

    if len(np.unique(final_order)) != m:
        raise ValueError("GO-BS-FC order is not a valid permutation.")

    return final_order


# ============================================================
# Fixed-order BSTabDiff training
# ============================================================
def train_bstabdiff_gobs_fc(X, y, params, fc_params, name):
    order = build_feature_clustered_order_gpu(
        X,
        num_clusters=fc_params["gobs_num_clusters"],
        metric=fc_params["gobs_metric"],
        bins=fc_params["gobs_bins"],
        top_k=fc_params["gobs_top_k"],
        refine_passes=fc_params["gobs_refine_passes"],
    )

    feature_specs = [
        FeatureSpec(name=f"f{j}", kind="continuous")
        for j in range(X.shape[1])
    ]

    blocks = make_contiguous_blocks_from_order(X.shape[1], params["M"])
    X_perm = X[:, order]
    feature_specs_perm = [feature_specs[j] for j in order]

    gen = fit_block_subunit_generator(
        X=X_perm,
        feature_specs=feature_specs_perm,
        y=y,
        M=params["M"],
        blocks=blocks,
        permute_features=False,
        prior_type=params["prior_type"],
        device=DEVICE,
        seed=GLOBAL_SEED,
        prior_epochs=params["prior_epochs"],
        prior_batch=params["prior_batch"],
        prior_lr=params["prior_lr"],
        verbose_every=0,
        save_dir=None,
        save_name=f"bstabdiff_energy_gobs_fc_{name}",
        save_best=True,
        use_ema=True,
        ema_decay=params["ema_decay"],
        return_train_info=False,

        # fixed external FC order; internal GO-BS off
        use_gobs=False,
        gobs_num_clusters=fc_params["gobs_num_clusters"],
        gobs_metric=fc_params["gobs_metric"],
        gobs_bins=fc_params["gobs_bins"],
        gobs_top_k=fc_params["gobs_top_k"],
        gobs_refine_order=False,
        gobs_direction_select=False,
        gobs_refine_passes=fc_params["gobs_refine_passes"],
        gobs_boundary_refine_passes=1,
        gobs_boundary_window=8,
        gobs_lambda_cross=1.0,
        gobs_gamma_balance=1e-6,
        gobs_use_cpu_kmeans=False,
    )

    return gen


def read_codecarbon_last_row(csv_path):
    df = pd.read_csv(csv_path)
    row = df.iloc[-1].to_dict()

    def get(*keys, default=np.nan):
        for k in keys:
            if k in row:
                return row[k]
        return default

    return {
        "energy_kwh": get("energy_consumed"),
        "emissions_kg_co2e": get("emissions"),
        "cpu_energy_kwh": get("cpu_energy"),
        "gpu_energy_kwh": get("gpu_energy"),
        "ram_energy_kwh": get("ram_energy"),
        "cpu_power_w": get("cpu_power"),
        "gpu_power_w": get("gpu_power"),
        "ram_power_w": get("ram_power"),
    }


# ============================================================
# Run CodeCarbon tracking
# ============================================================
set_seed(GLOBAL_SEED)

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.cuda.device_count():", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))

all_rows = []

for name, cfg in DATASETS.items():
    print(f"\n{'=' * 80}")
    print(f"Dataset: {name} | BSTabDiff + GO-BS-FC")
    print(f"{'=' * 80}")

    X, y, feature_names = load_dataset(cfg["csv"], cfg["target"])
    print(f"Loaded {name}: X={X.shape}, classes={np.unique(y)}")

    params = BEST_PARAMS[name]
    fc_params = GOBS_FC_PARAMS[name]
    cleanup()

    output_file = f"codecarbon_gobs_fc_{name}.csv"
    output_path = os.path.join(OUTPUT_DIR, output_file)

    tracker = EmissionsTracker(
        project_name=f"BSTabDiff_GO_BS_FC_{name}",
        output_dir=OUTPUT_DIR,
        output_file=output_file,
        measure_power_secs=1,
        save_to_file=True,
        log_level="error",
    )

    t0 = time.time()
    tracker.start()

    try:
        _ = train_bstabdiff_gobs_fc(X, y, params, fc_params, name)
    finally:
        emissions = tracker.stop()
        elapsed = time.time() - t0
        cleanup()

    cc = read_codecarbon_last_row(output_path)

    row = {
        "Dataset": name,
        "n": X.shape[0],
        "m": X.shape[1],
        "classes": len(np.unique(y)),
        "energy_kwh": cc["energy_kwh"],
        "emissions_kg_co2e": cc["emissions_kg_co2e"],
        "cpu_energy_kwh": cc["cpu_energy_kwh"],
        "gpu_energy_kwh": cc["gpu_energy_kwh"],
        "ram_energy_kwh": cc["ram_energy_kwh"],
        "cpu_power_w": cc["cpu_power_w"],
        "gpu_power_w": cc["gpu_power_w"],
        "ram_power_w": cc["ram_power_w"],
        "elapsed_sec": elapsed,
        "codecarbon_returned_emissions": emissions,
    }

    all_rows.append(row)

    print(
        f"{name}: energy={row['energy_kwh']:.6f} kWh, "
        f"CO2e={row['emissions_kg_co2e']:.6f} kg, "
        f"time={elapsed:.2f}s"
    )

out_df = pd.DataFrame(all_rows)
out_csv = os.path.join(OUTPUT_DIR, "bstabdiff_gobs_fc_energy_co2_summary.csv")
out_df.to_csv(out_csv, index=False)

print("\nSaved summary:")
print(out_csv)

try:
    display(out_df)
except Exception:
    print(out_df.to_markdown(index=False))

CUDA_VISIBLE_DEVICES: 6
torch.cuda.is_available(): True
torch.cuda.device_count(): 1
Using GPU: NVIDIA RTX A6000

Dataset: COL | BSTabDiff + GO-BS-FC
Loaded COL: X=(62, 2000), classes=[0 1]
COL: energy=0.000540 kWh, CO2e=0.000484 kg, time=11.31s

Dataset: LNG | BSTabDiff + GO-BS-FC
Loaded LNG: X=(203, 3312), classes=[0 1 2 3 4]
LNG: energy=0.000571 kWh, CO2e=0.000511 kg, time=12.49s

Dataset: GLI | BSTabDiff + GO-BS-FC
Loaded GLI: X=(85, 22283), classes=[0 1]
GLI: energy=0.006501 kWh, CO2e=0.005827 kg, time=242.98s

Dataset: SMK | BSTabDiff + GO-BS-FC
Loaded SMK: X=(187, 19993), classes=[0 1]
SMK: energy=0.003730 kWh, CO2e=0.003343 kg, time=104.87s

Dataset: AML | BSTabDiff + GO-BS-FC
Loaded AML: X=(72, 7129), classes=[0 1]
AML: energy=0.002380 kWh, CO2e=0.002133 kg, time=63.51s

Dataset: PRS | BSTabDiff + GO-BS-FC
Loaded PRS: X=(102, 5966), classes=[0 1]
PRS: energy=0.052860 kWh, CO2e=0.047373 kg, time=1143.65s

Dataset: ARC | BSTabDiff + GO-BS-FC
Loaded ARC: X=(200, 10000), classes=[

,Dataset,n,m,classes,energy_kwh,emissions_kg_co2e,cpu_energy_kwh,gpu_energy_kwh,ram_energy_kwh,cpu_power_w,gpu_power_w,ram_power_w,elapsed_sec,codecarbon_returned_emissions
0,COL,62,2000,2,0.000540,0.000484,0.000117,0.000212,0.000212,37.936518,94.435880,70.0,11.313164,0.000484
1,LNG,203,3312,5,0.000571,0.000511,0.000083,0.000237,0.000250,31.142372,106.961447,70.0,12.485962,0.000511
2,GLI,85,22283,2,0.006501,0.005827,0.001608,0.001976,0.002917,37.592738,45.459744,70.0,242.980098,0.005827
3,SMK,187,19993,2,0.003730,0.003343,0.000944,0.001052,0.001734,37.161673,56.726703,70.0,104.868680,0.003343
4,AML,72,7129,2,0.002380,0.002133,0.000423,0.001145,0.000812,35.786173,86.408101,70.0,63.509824,0.002133
5,PRS,102,5966,2,0.052860,0.047373,0.010111,0.008598,0.034151,22.726751,58469.244986,70.0,1143.649895,0.047373
6,ARC,200,10000,2,0.002028,0.001818,0.000480,0.000614,0.000934,36.304269,70.795212,70.0,45.316777,0.001818
7,TOX,171,5748,4,0.000813,0.000728,0.000209,0.000225,0.000378,37.404023,125.313044,70.0,12.193477,0.000728


# default/one fixed configurations

In [ ]:
# If needed:
# !pip install codecarbon psutil

import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

import gc
import time
import random
import warnings
import logging
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
logging.getLogger("codecarbon").setLevel(logging.ERROR)
logging.getLogger("pynvml").setLevel(logging.ERROR)
logging.getLogger("urllib3").setLevel(logging.ERROR)

import torch
from codecarbon import EmissionsTracker
from bstabdiff_gobs import FeatureSpec, fit_block_subunit_generator


GLOBAL_SEED = 42
DEVICE = "cuda:0"
OUTPUT_DIR = "bstabdiff_energy_codecarbon_default_nogobs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DATASETS = {
    "COL": {"csv": "coloncancer_encoded.csv", "target": "label"},
    "LNG": {"csv": "lung_combined_encoded.csv", "target": "Label"},
    "GLI": {"csv": "GLI-85_encoded.csv", "target": "label"},
    "SMK": {"csv": "SMK_CAN_187_combined_encoded.csv", "target": "Label"},
    "AML": {"csv": "ALLAML_combined_encoded.csv", "target": "Label"},
    "PRS": {"csv": "Prostate-GE_combined_encoded.csv", "target": "Label"},
    "ARC": {"csv": "arcene_combined_encoded.csv", "target": "Label"},
    "TOX": {"csv": "TOX171_combined_encoded.csv", "target": "Label"},
}

# Fixed/default BSTabDiff configuration
DEFAULT_PARAMS = {
    "COL": dict(M=32),
    "LNG": dict(M=64),
    "GLI": dict(M=128),
    "SMK": dict(M=192),
    "AML": dict(M=64),
    "PRS": dict(M=64),
    "ARC": dict(M=64),
    "TOX": dict(M=32),
}

FIXED_PRIOR_TYPE = "diffusion"
FIXED_PRIOR_EPOCHS = 10_000
FIXED_PRIOR_BATCH = 128
FIXED_PRIOR_LR = 1e-3
FIXED_USE_EMA = True
FIXED_EMA_DECAY = 0.999


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_dataset(csv_path, target_col):
    df = pd.read_csv(csv_path)

    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in {csv_path}.")

    y_raw = df[target_col].values
    X_df = df.drop(columns=[target_col]).select_dtypes(include=[np.number])

    X = X_df.to_numpy(dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    if y_raw.dtype.kind in {"U", "S", "O"}:
        y, _ = pd.factorize(y_raw)
    else:
        uniq = np.unique(y_raw)
        if not np.array_equal(uniq, np.arange(len(uniq))):
            y, _ = pd.factorize(y_raw)
        else:
            y = y_raw.astype(int)

    return X, y.astype(int), list(X_df.columns)


def train_bstabdiff_default_nogobs(X, y, params, name):
    feature_specs = [
        FeatureSpec(name=f"f{j}", kind="continuous")
        for j in range(X.shape[1])
    ]

    gen = fit_block_subunit_generator(
        X=X,
        feature_specs=feature_specs,
        y=y,
        M=params["M"],
        blocks=None,
        permute_features=False,
        prior_type=FIXED_PRIOR_TYPE,
        device=DEVICE,
        seed=GLOBAL_SEED,
        prior_epochs=FIXED_PRIOR_EPOCHS,
        prior_batch=FIXED_PRIOR_BATCH,
        prior_lr=FIXED_PRIOR_LR,
        verbose_every=0,
        save_dir=None,
        save_name=f"bstabdiff_energy_default_nogobs_{name}",
        save_best=True,
        use_ema=FIXED_USE_EMA,
        ema_decay=FIXED_EMA_DECAY,
        return_train_info=False,

        # no GO-BS / no permutation / no predefined blocks
        use_gobs=False,
        gobs_num_clusters=7,
        gobs_metric="correlation",
        gobs_bins=32,
        gobs_top_k=None,
        gobs_refine_order=False,
        gobs_direction_select=False,
        gobs_refine_passes=0,
        gobs_boundary_refine_passes=1,
        gobs_boundary_window=8,
        gobs_lambda_cross=1.0,
        gobs_gamma_balance=1e-6,
        gobs_use_cpu_kmeans=False,
    )
    return gen


def read_codecarbon_last_row(csv_path):
    df = pd.read_csv(csv_path)
    row = df.iloc[-1].to_dict()

    def get(*keys, default=np.nan):
        for k in keys:
            if k in row:
                return row[k]
        return default

    return {
        "energy_kwh": get("energy_consumed"),
        "emissions_kg_co2e": get("emissions"),
        "cpu_energy_kwh": get("cpu_energy"),
        "gpu_energy_kwh": get("gpu_energy"),
        "ram_energy_kwh": get("ram_energy"),
        "cpu_power_w": get("cpu_power"),
        "gpu_power_w": get("gpu_power"),
        "ram_power_w": get("ram_power"),
    }


set_seed(GLOBAL_SEED)

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.cuda.device_count():", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))

all_rows = []

for name, cfg in DATASETS.items():
    print(f"\n{'=' * 80}")
    print(f"Dataset: {name} | BSTabDiff default no-GO-BS")
    print(f"{'=' * 80}")

    X, y, feature_names = load_dataset(cfg["csv"], cfg["target"])
    print(f"Loaded {name}: X={X.shape}, classes={np.unique(y)}")

    params = DEFAULT_PARAMS[name]
    cleanup()

    output_file = f"codecarbon_default_nogobs_{name}.csv"
    output_path = os.path.join(OUTPUT_DIR, output_file)

    tracker = EmissionsTracker(
        project_name=f"BSTabDiff_default_no_GOBS_{name}",
        output_dir=OUTPUT_DIR,
        output_file=output_file,
        measure_power_secs=1,
        save_to_file=True,
        log_level="error",
    )

    t0 = time.time()
    tracker.start()

    try:
        _ = train_bstabdiff_default_nogobs(X, y, params, name)
    finally:
        emissions = tracker.stop()
        elapsed = time.time() - t0
        cleanup()

    cc = read_codecarbon_last_row(output_path)

    row = {
        "Dataset": name,
        "n": X.shape[0],
        "m": X.shape[1],
        "classes": len(np.unique(y)),
        "M": params["M"],
        "prior_type": FIXED_PRIOR_TYPE,
        "prior_epochs": FIXED_PRIOR_EPOCHS,
        "prior_batch": FIXED_PRIOR_BATCH,
        "prior_lr": FIXED_PRIOR_LR,
        "ema_decay": FIXED_EMA_DECAY,
        "energy_kwh": cc["energy_kwh"],
        "emissions_kg_co2e": cc["emissions_kg_co2e"],
        "cpu_energy_kwh": cc["cpu_energy_kwh"],
        "gpu_energy_kwh": cc["gpu_energy_kwh"],
        "ram_energy_kwh": cc["ram_energy_kwh"],
        "cpu_power_w": cc["cpu_power_w"],
        "gpu_power_w": cc["gpu_power_w"],
        "ram_power_w": cc["ram_power_w"],
        "elapsed_sec": elapsed,
        "codecarbon_returned_emissions": emissions,
    }

    all_rows.append(row)

    print(
        f"{name}: M={params['M']}, "
        f"energy={row['energy_kwh']:.6f} kWh, "
        f"CO2e={row['emissions_kg_co2e']:.6f} kg, "
        f"time={elapsed:.2f}s"
    )

out_df = pd.DataFrame(all_rows)
out_csv = os.path.join(OUTPUT_DIR, "bstabdiff_default_nogobs_energy_co2_summary.csv")
out_df.to_csv(out_csv, index=False)

print("\nSaved summary:")
print(out_csv)

try:
    display(out_df)
except Exception:
    print(out_df.to_markdown(index=False))

CUDA_VISIBLE_DEVICES: 6
torch.cuda.is_available(): True
torch.cuda.device_count(): 1
Using GPU: NVIDIA RTX A6000

Dataset: COL | BSTabDiff default no-GO-BS
Loaded COL: X=(62, 2000), classes=[0 1]
COL: M=32, energy=0.001414 kWh, CO2e=0.001267 kg, time=41.30s

Dataset: LNG | BSTabDiff default no-GO-BS
Loaded LNG: X=(203, 3312), classes=[0 1 2 3 4]
LNG: M=64, energy=0.001481 kWh, CO2e=0.001327 kg, time=38.23s

Dataset: GLI | BSTabDiff default no-GO-BS
Loaded GLI: X=(85, 22283), classes=[0 1]
GLI: M=128, energy=0.001643 kWh, CO2e=0.001473 kg, time=45.00s

Dataset: SMK | BSTabDiff default no-GO-BS
Loaded SMK: X=(187, 19993), classes=[0 1]
SMK: M=192, energy=0.001522 kWh, CO2e=0.001364 kg, time=43.14s

Dataset: AML | BSTabDiff default no-GO-BS
Loaded AML: X=(72, 7129), classes=[0 1]
AML: M=64, energy=0.001439 kWh, CO2e=0.001290 kg, time=40.69s

Dataset: PRS | BSTabDiff default no-GO-BS
Loaded PRS: X=(102, 5966), classes=[0 1]
PRS: M=64, energy=0.001618 kWh, CO2e=0.001450 kg, time=42.33s

Dat

,Dataset,n,m,classes,M,prior_type,prior_epochs,prior_batch,prior_lr,ema_decay,energy_kwh,emissions_kg_co2e,cpu_energy_kwh,gpu_energy_kwh,ram_energy_kwh,cpu_power_w,gpu_power_w,ram_power_w,elapsed_sec,codecarbon_returned_emissions
0,COL,62,2000,2,32,diffusion,10000,128,0.001,0.999,0.001414,0.001267,0.000188,0.000870,0.000355,37.128628,97.112784,70.00,41.303503,0.001267
1,LNG,203,3312,5,64,diffusion,10000,128,0.001,0.999,0.001481,0.001327,0.000226,0.000830,0.000425,37.139155,108.031537,70.00,38.231873,0.001327
2,GLI,85,22283,2,128,diffusion,10000,128,0.001,0.999,0.001643,0.001473,0.000236,0.000964,0.000444,37.122842,110.416255,70.00,44.996615,0.001473
3,SMK,187,19993,2,192,diffusion,10000,128,0.001,0.999,0.001522,0.001364,0.000206,0.000929,0.000387,37.113746,112.044765,70.00,43.140766,0.001364
4,AML,72,7129,2,64,diffusion,10000,128,0.001,0.999,0.001439,0.001290,0.000199,0.000847,0.000393,36.187483,100.758718,70.00,40.693311,0.001290
5,PRS,102,5966,2,64,diffusion,10000,128,0.001,0.999,0.001618,0.001450,0.000255,0.000882,0.000481,38.055430,114.109055,71.75,42.334356,0.001450
6,ARC,200,10000,2,64,diffusion,10000,128,0.001,0.999,0.001546,0.001385,0.000221,0.000910,0.000416,37.138282,111.178950,70.00,41.598882,0.001385
7,TOX,171,5748,4,32,diffusion,10000,128,0.001,0.999,0.001545,0.001384,0.000227,0.000890,0.000427,37.120808,109.992820,70.00,40.849005,0.001384
